# 05_tuning_lightgbm.ipynb
- 노트북의 목표
    - LightGBM 모델 튜닝. 30% 샘플 데이터로 RandomizedSearchCV를 실행한 뒤 찾은 최적 파라미터로 전체 데이터에 재학습

## 목차
1. 환경 세팅 및 데이터 로드
2. 30% 샘플링
3. RandomizedSearchCV 튜닝
4. 전체 데이터로 최종 모델 학습
5. 최종 평가
6. 피처 중요도 분석
7. v2 튜닝 결과 대비 성능 변화 확인
8. 결과 저장

### 1. 환경 세팅 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import time, os

from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, roc_curve, classification_report
)

plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:.4f}'.format
print('임포트 완료')


In [ ]:
X_train = pd.read_csv('../data/X_train_v3.csv')
X_val   = pd.read_csv('../data/X_val_v3.csv')
y_train = pd.read_csv('../data/y_train_v3.csv').squeeze()
y_val   = pd.read_csv('../data/y_val_v3.csv').squeeze()

# v2 튜닝 결과 로드 (비교용)
try:
    v2_tuned = pd.read_csv('../results/lightgbm_tuned_v2.csv')
    v2_auc = v2_tuned['auc_roc'].values[0]
    print(f'v2 튜닝 AUC (비교 기준): {v2_auc}')
except:
    v2_auc = None
    print('v2 튜닝 결과 없음 — v3 베이스라인과 비교')

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}')


### 2. 30% 샘플링

In [ ]:
# 전체 train 데이터의 30%만 사용해 튜닝 속도를 높입니다.
# stratify=y_train으로 이탈 비율을 유지합니다.
X_sample, _, y_sample, _ = train_test_split(
    X_train, y_train,
    train_size=0.3,
    random_state=42,
    stratify=y_train
)

print(f'샘플 데이터: {X_sample.shape} ({len(X_sample)/len(X_train)*100:.0f}%)')
print(f'샘플 이탈률: {y_sample.mean():.4f}')


### 3. RandomizedSearchCV 튜닝

In [ ]:
param_grid = {
    'num_leaves'       : [31, 63, 127, 255],
    'max_depth'        : [-1, 10, 15, 20],
    'learning_rate'    : [0.03, 0.05, 0.1],
    'n_estimators'     : [200, 300, 400],
    'subsample'        : [0.7, 0.8, 0.9],
    'colsample_bytree' : [0.7, 0.8, 0.9],
    'min_child_samples': [20, 50, 100]
}

base_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)

# n_iter=20: 전체 조합 중 20개만 무작위 탐색
# cv=2: 검증 fold 2개로 속도 최적화
search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=20,
    scoring='roc_auc',
    cv=2,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

start = time.time()
search.fit(X_sample, y_sample)
tuning_time = time.time() - start

best_params = search.best_params_
print(f'튜닝 소요 시간: {tuning_time:.1f}초 (30% 샘플 기준)')
print('--- 최적 파라미터 ---')
print(best_params)
print(f'CV 최고 AUC: {search.best_score_:.4f}')


### 4. 전체 데이터로 최종 모델 학습

In [ ]:
# 30% 샘플로 찾은 파라미터를 전체 train 데이터(100%)로 재학습
final_model = lgb.LGBMClassifier(**best_params, random_state=42, n_jobs=-1, verbose=-1)
start = time.time()
final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
)
train_time = time.time() - start

print(f'최종 학습 완료 ({train_time:.1f}초)')


### 5. 최종 평가

In [ ]:
y_pred  = final_model.predict(X_val)
y_proba = final_model.predict_proba(X_val)[:, 1]

final_acc = accuracy_score(y_val, y_pred)
final_auc = roc_auc_score(y_val, y_proba)
final_f1  = f1_score(y_val, y_pred)

print('--- LightGBM v3 최종 성능 ---')
print(f'Accuracy : {final_acc:.4f}')
print(f'AUC-ROC  : {final_auc:.4f}')
print(f'F1-Score : {final_f1:.4f}')
print()
print(classification_report(y_val, y_pred, target_names=['유지(0)','이탈(1)']))


In [ ]:
cm = confusion_matrix(y_val, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['유지(0)','이탈(1)'], yticklabels=['유지(0)','이탈(1)'], ax=axes[0])
axes[0].set_title('LightGBM v3 Confusion Matrix')
axes[0].set_ylabel('실제값'); axes[0].set_xlabel('예측값')

fpr, tpr, _ = roc_curve(y_val, y_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'LightGBM v3 (AUC={final_auc:.4f})')
axes[1].plot([0,1],[0,1], linestyle='--', color='gray', label='Random Baseline')
axes[1].set_title('LightGBM v3 ROC Curve')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend()

plt.tight_layout(); plt.show()


### 6. 피처 중요도 분석

In [ ]:
all_imp = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)
all_imp['rank'] = all_imp.index + 1

# 상위 20개 시각화
plt.figure(figsize=(10, 7))
sns.barplot(x='importance', y='feature', data=all_imp.head(20), palette='Blues_r')
plt.title('LightGBM v3 Feature Importance (Top 20)')
plt.tight_layout(); plt.show()

# 시퀀스 피처 순위 확인
seq_cols = ['interval_t1','interval_t2','interval_t3','interval_t4','interval_t5']
print('--- 시퀀스 피처(interval_t1~t5) 전체 순위 ---')
print(all_imp[all_imp['feature'].isin(seq_cols)][['rank','feature','importance']].to_string(index=False))


### 7. v2 튜닝 결과 대비 성능 변화 확인

In [ ]:
if v2_auc is not None:
    improvement = final_auc - v2_auc
    print(f'--- LightGBM v2 튜닝 vs v3 최종 비교 ---')
    print(f'v2 튜닝 AUC : {v2_auc:.4f}')
    print(f'v3 최종 AUC : {final_auc:.4f}')
    print(f'개선폭       : {improvement:+.4f}')
    if improvement < 0.003:
        print('\n→ 시퀀스 피처 추가에도 성능 향상이 미미합니다.')
        print('   RFM 기반 집계 피처만으로도 이 데이터의 이탈 패턴을 충분히 설명하고 있음을 시사합니다.')
    else:
        print('\n→ 시퀀스 피처가 성능 개선에 기여했습니다.')
else:
    print('v2 결과가 없어 v3 베이스라인과 비교하세요.')


### 8. 결과 저장

In [ ]:
os.makedirs('../results', exist_ok=True)

result = pd.DataFrame([{
    'model'      : 'LightGBM_v3_tuned',
    'accuracy'   : round(final_acc, 4),
    'auc_roc'    : round(final_auc, 4),
    'f1_score'   : round(final_f1, 4),
    'train_time' : round(train_time, 1),
    'v2_auc'     : v2_auc,
    'improvement': round(final_auc - v2_auc, 4) if v2_auc else None
}])

result.to_csv('../results/lightgbm_tuned_v3.csv', index=False)
print('결과 저장 완료')
print(result.to_string(index=False))
